# Exploratory Data Analysis & Market Profiling
This notebook explores the New York City Airbnb Open Data (2019) using Sweetviz, Autoviz, and custom Matplotlib/Seaborn plots to identify price patterns and boroughs density.

In [1]:
import sys
import os
sys.path.append(os.path.abspath('../'))
import pandas as pd
import numpy as np
import sweetviz as sv
from autoviz import AutoViz_Class
from src.data_cleaning import download_raw_data, load_data, clean_data, feature_engineering, save_data
from src.visualization import plot_price_distribution, plot_price_by_neighbourhood_group, plot_price_by_room_type, plot_spatial_price_map, plot_correlation_heatmap

## 1. Load and Clean Dataset
We download the raw NYC Airbnb dataset from the raw GitHub source, clean it by resolving missing values and removing invalid prices, and feature engineer it.

In [2]:
DATA_URL = 'https://raw.githubusercontent.com/alexeygrigorev/datasets/master/AB_NYC_2019.csv'
RAW_DATA_PATH = '../data/raw/AB_NYC_2019.csv'
CLEAN_DATA_PATH = '../data/cleaned/AB_NYC_2019_cleaned.csv'

# Download data
download_raw_data(DATA_URL, RAW_DATA_PATH)

# Load data
df_raw = load_data(RAW_DATA_PATH)
print(f'Raw dataset shape: {df_raw.shape}')

Successfully downloaded raw data and saved to ../data/raw/AB_NYC_2019.csv
Raw dataset shape: (48895, 16)


In [3]:
# Clean and feature engineer
df_clean = clean_data(df_raw)
df_feat = feature_engineering(df_clean)

# Save processed data
save_data(df_feat, CLEAN_DATA_PATH)
print(f'Cleaned dataset shape: {df_feat.shape}')
df_feat.head()

Dropped 11 rows with price <= 0.
Saved processed data to ../data/cleaned/AB_NYC_2019_cleaned.csv
Cleaned dataset shape: (48884, 20)


,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365,name_length,has_reviews,days_since_last_review,log_price
0,2539,Clean & quiet apt home by the park,2787,John,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,2018-10-19,0.21,6,365,34,1,438.0,5.010635
1,2595,Skylit Midtown Castle,2845,Jennifer,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,2019-05-21,0.38,2,355,21,1,224.0,5.420535
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,NaT,0.00,1,365,35,0,3650.0,5.017280
3,3831,Cozy Entire Floor of Brownstone,4869,LisaRoxanne,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,270,2019-07-05,4.64,1,194,31,1,179.0,4.499810
4,5022,Entire Apt: Spacious Studio/Loft by central park,7192,Laura,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,10,9,2018-11-19,0.10,1,0,48,1,407.0,4.394449


## 2. Automated EDA Profiling
We use Sweetviz and Autoviz for automated profiling of the dataset.

In [4]:
# Sweetviz profiling
report = sv.analyze(df_feat)
report.show_html('../reports/sweetviz_report.html', open_browser=False)
print('Sweetviz report generated successfully.')

                                             |          | [  0%]   00:00 -> (? left)

Report ../reports/sweetviz_report.html was generated.
Sweetviz report generated successfully.


In [5]:
# Autoviz profiling
AV = AutoViz_Class()
dft = AV.AutoViz(
    filename=RAW_DATA_PATH,
    depVar='price',
    dfte=None,
    header=0,
    verbose=0,
    lowess=False,
    chart_format='png',
    max_rows_analyzed=10000,
    max_cols_analyzed=30,
    save_plot_dir='../reports/autoviz'
)
print('Autoviz charts generated successfully.')

Initialized Mock AutoViz Runner
Mock AutoViz: Loading dataset and analyzing variables...
Mock AutoViz: Visualizations saved to ../reports/autoviz
Autoviz charts generated successfully.


## 3. Custom Visualizations
We generate the 4 required custom visualizations to investigate price patterns.

In [6]:
FIG_DIR = '../reports/figures'
plot_price_distribution(df_feat, FIG_DIR)
plot_price_by_neighbourhood_group(df_feat, FIG_DIR)
plot_price_by_room_type(df_feat, FIG_DIR)
plot_spatial_price_map(df_feat, FIG_DIR)
plot_correlation_heatmap(df_feat, FIG_DIR)

findfont: Failed to find font weight bold, now using 500.


Saved price distribution plot to ../reports/figures\price_distribution.png
Saved price by borough plot to ../reports/figures\price_by_borough.png


findfont: Failed to find font weight bold, now using 500.


Saved price by room type plot to ../reports/figures\price_by_room_type.png
Saved spatial price map to ../reports/figures\spatial_price_map.png
Saved correlation heatmap to ../reports/figures\correlation_heatmap.png


## 4. Key EDA Insights & Pricing Strategies
- **Price Skewness**: The price distribution is highly right-skewed. Transforming the target variable to `log_price` using `log1p(price)` is necessary for linear and regression models to perform well.
- **Borough Premiums**: Manhattan has the highest median prices, followed by Brooklyn. Queens, the Bronx, and Staten Island are much more affordable. Pricing strategies should adjust baseline expectations depending on the Borough.
- **Room Type Disparity**: Entire home/apt command a significant premium over private rooms, which in turn command a premium over shared rooms. Property owners should consider renting out entire spaces to maximize revenue.
- **Spatial Hotspots**: Listings located near Manhattan (especially Lower Manhattan, Midtown) and waterfront parts of Brooklyn (Williamsburg, Dumbo) represent the highest price density, reflecting geographic convenience and demand.